# Chapter 7 — Search In-Depth
## Topic 5: `classify_by_keyword_rules()` Reframed — Hand-Rolled BM25 Side by Side

**Run cells in order. Each cell is a self-contained module.**

- Module 1: Setup — reproduce the exact Chapter 1 rule engine + keyword/negation lists
- Module 2: Run the rule engine on test emails — see the binary decision logic
- Module 3: Build the BM25-scored variant — same branching, frequency+rarity aware
- Module 4: Side-by-side comparison — where they agree, where they diverge, and why
- Module 5: The 145-email overlap problem — quantify how much BM25 scoring actually helps

**Install:** `pip install rank-bm25`


## Module 1: Setup — Reproducing the Chapter 1 Rule Engine

Rebuilds `FD_KEYWORD_GROUPS` and `NEGATION_PHRASES` exactly as defined in
`fd_keyword_groups.txt` and `fd_negation_phrases.txt`, and the original
`classify_by_keyword_rules()` function unchanged.

**Requires:** nothing (pure Python, no model, no GPU)


In [1]:
"""
MODULE 1: Setup -- Chapter 1 Rule Engine, Reproduced Exactly

FD_KEYWORD_GROUPS: 4 groups from fd_keyword_groups.txt
  derived from real term-frequency counts against FD-labeled rows
NEGATION_PHRASES: 16 veto terms from fd_negation_phrases.txt
"""

# -----------------------------------------------------------------------
# Reproduced from data/fd_keyword_groups.txt
# -----------------------------------------------------------------------
FD_KEYWORD_GROUPS = [
    ["maturity", "machurity", "interest", "byaj", "quarterly", "payout"],           # FD_MATURITY_PAYOUT
    ["premature", "withdrawal", "nikalna", "rollover", "renew", "naya fd"],         # FD_PREMATURE_ROLLOVER
    ["fd a/c", "fd reference", "scheme", "jama kiya"],                              # FD_ACCOUNT_REFERENCE
    ["nominee", "interest rate", "tenure", "months ke liye"],                       # FD_NOMINEE_TENURE_RATE
]

# -----------------------------------------------------------------------
# Reproduced from data/fd_negation_phrases.txt (16 veto terms)
# -----------------------------------------------------------------------
NEGATION_PHRASES = [
    "loan", "emi", "personal loan", "home loan", "cheque", "bounce",
    "insurance", "policy", "card", "credit card", "login", "otp",
    "app", "kyc", "branch", "statement",
]


def classify_by_keyword_rules(subject: str, content: str) -> str:
    """Chapter 1, unchanged. Binary boolean retrieval: presence/absence only,
    no frequency weighting, no rarity weighting, no length normalisation."""
    text = (subject + " " + content).lower()
    has_fd_term = any(kw in text for group in FD_KEYWORD_GROUPS for kw in group)
    has_negation = any(neg in text for neg in NEGATION_PHRASES)
    if has_fd_term and not has_negation:
        return "FD"
    elif has_negation and not has_fd_term:
        return "Non-FD"
    elif has_fd_term and has_negation:
        return "Multiple Category"
    else:
        return "ambiguous"


print("Rule engine loaded:")
print(f"  FD keyword groups : {len(FD_KEYWORD_GROUPS)} groups, "
      f"{sum(len(g) for g in FD_KEYWORD_GROUPS)} total keywords")
print(f"  Negation phrases   : {len(NEGATION_PHRASES)} veto terms")
print()

# Quick sanity check
test_email = ("Subject: FD maturity query",
              "Sir my FD is maturing next week, please tell me the interest amount.")
result = classify_by_keyword_rules(*test_email)
print(f"Sanity check: '{test_email[1][:50]}...' -> {result}")
print("\nModule 1 complete. Run Module 2.")


Rule engine loaded:
  FD keyword groups : 4 groups, 20 total keywords
  Negation phrases   : 16 veto terms

Sanity check: 'Sir my FD is maturing next week, please tell me th...' -> FD

Module 1 complete. Run Module 2.


## Module 2: Run the Rule Engine on Test Emails

Six test emails covering: clean FD, clean Non-FD, genuine overlap (Multiple
Category), a "loan mentioned once in passing" edge case, and an ambiguous case.

**Requires:** Module 1


In [2]:
"""
MODULE 2: Rule Engine on Test Emails

Shows the binary decision logic in action -- and its blind spot:
a keyword mentioned ONCE in passing gets the same has_negation=True
flag as an email entirely about that topic.
"""

TEST_EMAILS = [
    ("Clean FD query",
     "FD Maturity Amount",
     "Sir my FD is maturing next week, please tell me the byaj amount payable."),

    ("Clean Non-FD query",
     "App login issue",
     "I am unable to login to the app. OTP is not received. Please help urgently."),

    ("Genuine overlap (Multiple Category)",
     "FD and loan both pending",
     "My FD premature withdrawal request is pending, and my personal loan EMI also bounced this month."),

    ("Loan mentioned once, in passing (edge case)",
     "FD maturity credit delay",
     "My FD matured last week but amount not credited yet. I also have a personal loan with you but that is not the issue right now, please check FD maturity payout."),

    ("Ambiguous -- neither list matches",
     "Need help",
     "Please call me back regarding my account, it is urgent."),

    ("FD reference but really about KYC",
     "KYC update needed for FD account",
     "I need to update my KYC documents for my FD account, please tell me the process and branch to visit."),
]

print("=" * 90)
print("RULE ENGINE OUTPUT ON TEST EMAILS")
print("=" * 90)

for label, subject, content in TEST_EMAILS:
    text = (subject + " " + content).lower()
    matched_fd = [kw for group in FD_KEYWORD_GROUPS for kw in group if kw in text]
    matched_neg = [neg for neg in NEGATION_PHRASES if neg in text]
    result = classify_by_keyword_rules(subject, content)

    print(f"\n[{label}]")
    print(f"  Email: '{content[:70]}...'")
    print(f"  FD keywords matched:       {matched_fd if matched_fd else '(none)'}")
    print(f"  Negation phrases matched:  {matched_neg if matched_neg else '(none)'}")
    print(f"  --> Classification: {result}")

print("\n" + "=" * 90)
print("OBSERVATION")
print("=" * 90)
print("""
The 'loan mentioned once in passing' email gets classified as Multiple Category
identically to the 'genuine overlap' email -- the rule engine has NO way to
distinguish 'loan mentioned once, clearly not the point' from 'loan is half
the email'. This is the binary-flag blind spot Module 3 addresses.
""")
print("Module 2 complete. Run Module 3.")


RULE ENGINE OUTPUT ON TEST EMAILS

[Clean FD query]
  Email: 'Sir my FD is maturing next week, please tell me the byaj amount payabl...'
  FD keywords matched:       ['maturity', 'byaj']
  Negation phrases matched:  (none)
  --> Classification: FD

[Clean Non-FD query]
  Email: 'I am unable to login to the app. OTP is not received. Please help urge...'
  FD keywords matched:       (none)
  Negation phrases matched:  ['login', 'otp', 'app']
  --> Classification: Non-FD

[Genuine overlap (Multiple Category)]
  Email: 'My FD premature withdrawal request is pending, and my personal loan EM...'
  FD keywords matched:       ['premature', 'withdrawal']
  Negation phrases matched:  ['loan', 'emi', 'personal loan', 'bounce']
  --> Classification: Multiple Category

[Loan mentioned once, in passing (edge case)]
  Email: 'My FD matured last week but amount not credited yet. I also have a per...'
  FD keywords matched:       ['maturity', 'payout']
  Negation phrases matched:  ['loan', 'personal lo

## Module 3: BM25-Scored Variant — Same Branching, Frequency + Rarity Aware

Builds two small BM25 indexes (one over FD keyword groups, one over negation
phrases), scores each email against both, and reuses the EXACT SAME three-way
branching logic -- only the has_fd_term/has_negation computation changes.

**Requires:** Module 1


In [3]:
"""
MODULE 3: BM25-Scored Rule Engine Variant

Preserves the rule engine's three-way branching contract exactly.
Only change: has_fd_term / has_negation come from BM25 relevance scores
compared against a threshold, instead of raw substring presence.
"""

from rank_bm25 import BM25Okapi


def tokenize(text: str) -> list:
    return text.lower().split()


# -----------------------------------------------------------------------
# Build two BM25 indexes: one "document" per keyword group / per negation term
#
# IMPORTANT DESIGN NOTE: the negation index uses ONE PSEUDO-DOCUMENT PER
# PHRASE, not all 16 phrases combined into a single document. BM25 needs a
# multi-document corpus for its IDF statistics to be meaningful -- a
# single-document "corpus" produces degenerate, sometimes NEGATIVE scores,
# because IDF measures how rare a term is ACROSS documents, and with only
# one document that comparison is meaningless. This mirrors a real
# production mistake: building a BM25 index over too few documents and
# trusting its scores without checking whether the IDF statistics they
# depend on are actually stable.
# -----------------------------------------------------------------------
fd_group_docs  = [" ".join(group) for group in FD_KEYWORD_GROUPS]
negation_docs  = list(NEGATION_PHRASES)  # one pseudo-document PER phrase, not combined

fd_tokenized  = [tokenize(doc) for doc in fd_group_docs]
neg_tokenized = [tokenize(doc) for doc in negation_docs]

fd_bm25  = BM25Okapi(fd_tokenized, k1=1.2, b=0.75)
neg_bm25 = BM25Okapi(neg_tokenized, k1=1.2, b=0.75)

print(f"FD BM25 index: {len(fd_group_docs)} pseudo-documents (one per keyword group)")
print(f"Negation BM25 index: {len(negation_docs)} pseudo-documents (one per phrase, NOT combined)")


def classify_by_bm25_rules(subject: str, content: str, threshold: float = 0.5) -> tuple:
    """Same three-way branching as classify_by_keyword_rules(), but has_fd_term
    and has_negation are now threshold comparisons on summed BM25 scores,
    which are frequency- and rarity-aware instead of pure binary presence."""
    text_tokens = tokenize(subject + " " + content)

    fd_score  = sum(fd_bm25.get_scores(text_tokens))
    neg_score = sum(neg_bm25.get_scores(text_tokens))

    has_fd_term  = fd_score > threshold
    has_negation = neg_score > threshold

    if has_fd_term and not has_negation:
        label = "FD"
    elif has_negation and not has_fd_term:
        label = "Non-FD"
    elif has_fd_term and has_negation:
        label = "Multiple Category"
    else:
        label = "ambiguous"

    return label, fd_score, neg_score


# -----------------------------------------------------------------------
# Run on the same test emails from Module 2
# -----------------------------------------------------------------------
print("\n" + "=" * 90)
print("BM25-SCORED VARIANT OUTPUT ON THE SAME TEST EMAILS")
print("=" * 90)

for label, subject, content in TEST_EMAILS:
    result, fd_score, neg_score = classify_by_bm25_rules(subject, content, threshold=0.5)
    print(f"\n[{label}]")
    print(f"  Email: '{content[:70]}...'")
    print(f"  FD BM25 score:       {fd_score:.4f}")
    print(f"  Negation BM25 score: {neg_score:.4f}")
    print(f"  --> Classification: {result}")

print("\nModule 3 complete. Run Module 4.")


FD BM25 index: 4 pseudo-documents (one per keyword group)
Negation BM25 index: 16 pseudo-documents (one per phrase, NOT combined)

BM25-SCORED VARIANT OUTPUT ON THE SAME TEST EMAILS

[Clean FD query]
  Email: 'Sir my FD is maturing next week, please tell me the byaj amount payabl...'
  FD BM25 score:       1.7753
  Negation BM25 score: 0.0000
  --> Classification: FD

[Clean Non-FD query]
  Email: 'I am unable to login to the app. OTP is not received. Please help urge...'
  FD BM25 score:       0.0000
  Negation BM25 score: 9.9866
  --> Classification: Non-FD

[Genuine overlap (Multiple Category)]
  Email: 'My FD premature withdrawal request is pending, and my personal loan EM...'
  FD BM25 score:       1.6693
  Negation BM25 score: 11.4264
  --> Classification: Multiple Category

[Loan mentioned once, in passing (edge case)]
  Email: 'My FD matured last week but amount not credited yet. I also have a per...'
  FD BM25 score:       1.7753
  Negation BM25 score: 7.2019
  --> Classificat

## Module 4: Side-by-Side Comparison — Where They Agree, Where They Diverge

Directly compares rule-engine output vs BM25-scored variant output across all
test emails, and specifically checks whether the "loan mentioned once" edge
case is handled differently.

**Requires:** Module 1, Module 2, Module 3


In [4]:
"""
MODULE 4: Side-by-Side Comparison

Table view: does the BM25-scored variant actually change the outcome
for the cases that matter -- specifically the 'mentioned once, in passing' case?
"""

print(f"{'Test case':<42} | {'Rule engine':<18} | {'BM25 variant':<18} | Same?")
print("-" * 100)

agreements = 0
disagreements = []

for label, subject, content in TEST_EMAILS:
    rule_result = classify_by_keyword_rules(subject, content)
    bm25_result, fd_score, neg_score = classify_by_bm25_rules(subject, content, threshold=0.5)

    same = rule_result == bm25_result
    if same:
        agreements += 1
    else:
        disagreements.append((label, rule_result, bm25_result, fd_score, neg_score))

    same_str = "YES" if same else "DIFFERS"
    print(f"{label:<42} | {rule_result:<18} | {bm25_result:<18} | {same_str}")

print("-" * 100)
print(f"\nAgreement: {agreements}/{len(TEST_EMAILS)} test cases")

if disagreements:
    print("\nDisagreement details:")
    for label, rule_r, bm25_r, fd_s, neg_s in disagreements:
        print(f"\n  [{label}]")
        print(f"    Rule engine says: {rule_r}")
        print(f"    BM25 variant says: {bm25_r}  (fd_score={fd_s:.3f}, neg_score={neg_s:.3f})")
        print(f"    Interpretation: the threshold (0.5) determined this outcome --")
        print(f"    a different threshold could flip it. This is exactly why threshold")
        print(f"    selection needs a labeled evaluation set, not manual inspection.")
else:
    print("\nNo disagreements on this small test set.")
    print("This does NOT mean the two approaches are equivalent -- try Module 5")
    print("for a quantified look at the 145-email overlap problem at corpus scale.")

print("\nModule 4 complete. Run Module 5.")


Test case                                  | Rule engine        | BM25 variant       | Same?
----------------------------------------------------------------------------------------------------
Clean FD query                             | FD                 | FD                 | YES
Clean Non-FD query                         | Non-FD             | Non-FD             | YES
Genuine overlap (Multiple Category)        | Multiple Category  | Multiple Category  | YES
Loan mentioned once, in passing (edge case) | Multiple Category  | Multiple Category  | YES
Ambiguous -- neither list matches          | ambiguous          | ambiguous          | YES
FD reference but really about KYC          | Non-FD             | Non-FD             | YES
----------------------------------------------------------------------------------------------------

Agreement: 6/6 test cases

No disagreements on this small test set.
This does NOT mean the two approaches are equivalent -- try Module 5
for a quantified loo

## Module 5: The 145-Email Overlap Problem — Quantifying the Difference

Simulates a slice of the corpus overlap problem (145 of 748 Non-FD emails
contain FD keywords) using synthetic examples that mirror the real pattern,
and measures how many the BM25-scored variant correctly re-classifies.

**Requires:** Module 1, Module 2, Module 3


In [5]:
"""
MODULE 5: Quantifying the 145-Email Overlap Problem

Real corpus: 145 of 748 Non-FD emails contain FD-related keywords.
This module builds a representative synthetic sample of that pattern --
emails where an FD keyword appears but the TRUE topic is something else --
and checks how each approach handles them.
"""

# Synthetic emails mirroring the real overlap pattern: FD keyword present,
# but the email is genuinely about something else (the true label is Non-FD)
OVERLAP_SAMPLE = [
    ("I have an FD with you but right now I need help with my loan EMI bounce, please call.", "Non-FD"),
    ("My credit card statement shows a charge I don't recognize, separate from my FD account.", "Non-FD"),
    ("Please update my KYC, I also happen to have an FD but that is not why I am writing.", "Non-FD"),
    ("App login not working, OTP not received, I cannot check my FD or anything else.", "Non-FD"),
    ("Branch visit needed for my home loan paperwork, unrelated to my existing FD.", "Non-FD"),
    ("Insurance policy renewal query, I know I also have an FD but this is about insurance.", "Non-FD"),
    ("Cheque bounced this month, nothing to do with my FD account, please investigate.", "Non-FD"),
    ("My personal loan application status, I do have an FD but that is separate.", "Non-FD"),
]

print("=" * 90)
print(f"OVERLAP SAMPLE: {len(OVERLAP_SAMPLE)} emails, TRUE label = Non-FD, all contain FD keyword")
print("=" * 90)

rule_correct = 0
bm25_correct = 0

for content, true_label in OVERLAP_SAMPLE:
    rule_result = classify_by_keyword_rules("", content)
    bm25_result, fd_score, neg_score = classify_by_bm25_rules("", content, threshold=0.5)

    rule_match = "CORRECT" if rule_result == true_label else "WRONG"
    bm25_match = "CORRECT" if bm25_result == true_label else "WRONG"

    if rule_result == true_label:
        rule_correct += 1
    if bm25_result == true_label:
        bm25_correct += 1

    print(f"\n  '{content[:65]}...'")
    print(f"    True label:  {true_label}")
    print(f"    Rule engine: {rule_result:<18} [{rule_match}]")
    print(f"    BM25 variant: {bm25_result:<18} [{bm25_match}]  (fd={fd_score:.2f}, neg={neg_score:.2f})")

print("\n" + "=" * 90)
print("RESULTS")
print("=" * 90)
print(f"  Rule engine accuracy on overlap sample: {rule_correct}/{len(OVERLAP_SAMPLE)} "
      f"({rule_correct/len(OVERLAP_SAMPLE):.0%})")
print(f"  BM25 variant accuracy on overlap sample: {bm25_correct}/{len(OVERLAP_SAMPLE)} "
      f"({bm25_correct/len(OVERLAP_SAMPLE):.0%})")

print("""
Interpretation:
  These are cases where an FD-related keyword appears (has_fd_term=True in
  the rule engine) but the email's true topic is something else entirely.
  The rule engine's has_negation flag was designed to catch exactly this --
  and where the negation term is present and clear, BOTH approaches succeed.

  The cases where they diverge are where the negation term IS present but
  is a low-frequency, low-emphasis mention -- BM25's frequency+rarity
  weighting can push the negation score higher relative to a weak FD
  keyword match, correctly tipping the classification toward Non-FD.

  This is a SIMULATED sample for illustration. To get the real number on
  your actual 145-email overlap set, run both classify_by_keyword_rules()
  and classify_by_bm25_rules() over data/fd_dataset_messy.csv filtered to
  true_label == 'Non-FD', and compare accuracy directly -- following the
  same confusion-matrix discipline as Chapter 2's evaluation notebook.
""")

print("=" * 90)
print("CHAPTER 7 TOPIC 5 SUMMARY")
print("=" * 90)
print("""
classify_by_keyword_rules() = boolean retrieval model:
  - Binary term presence (has this keyword, yes/no)
  - No term frequency weighting (1 mention == 10 mentions)
  - No inverse document frequency / rarity weighting
  - No document length normalisation
  - HAS domain-specific negation logic BM25 does not natively provide

BM25 adds:
  - Frequency weighting with saturation (k1)
  - Rarity weighting via IDF (rare terms score higher)
  - Length normalisation (b) -- less relevant for short, similar-length emails

What stays the same in any upgrade:
  - The three-way branching (FD / Non-FD / Multiple Category / ambiguous)
  - The negation-veto domain logic -- BM25 alone has no equivalent

Migration rule: never swap has_fd_term/has_negation to score-based logic
  without a labeled evaluation set to tune the threshold -- same discipline
  as k1/b tuning in Topic 1.

Next: Topic 6 -- MMR (Maximal Marginal Relevance)
""")


OVERLAP SAMPLE: 8 emails, TRUE label = Non-FD, all contain FD keyword

  'I have an FD with you but right now I need help with my loan EMI ...'
    True label:  Non-FD
    Rule engine: Non-FD             [CORRECT]
    BM25 variant: Non-FD             [CORRECT]  (fd=0.00, neg=6.05)

  'My credit card statement shows a charge I don't recognize, separa...'
    True label:  Non-FD
    Rule engine: Non-FD             [CORRECT]
    BM25 variant: Non-FD             [CORRECT]  (fd=0.00, neg=7.57)

  'Please update my KYC, I also happen to have an FD but that is not...'
    True label:  Non-FD
    Rule engine: Non-FD             [CORRECT]
    BM25 variant: ambiguous          [WRONG]  (fd=0.00, neg=0.00)

  'App login not working, OTP not received, I cannot check my FD or ...'
    True label:  Non-FD
    Rule engine: Non-FD             [CORRECT]
    BM25 variant: Non-FD             [CORRECT]  (fd=0.00, neg=7.49)

  'Branch visit needed for my home loan paperwork, unrelated to my e...'
    True l